In [ ]:
# =====================
# PARAMETER
# =====================
file_excel = file_path
target_kdkab = "14"                                                               # code of the regency/city to be calculated

# =====================
# READ DATA
# =====================
df = pd.read_excel(file_excel, sheet_name="logikadasar")
df_pml = pd.read_excel(file_excel, sheet_name="PML")

In [ ]:
import pandas as pd
import os

# =====================
# PARAMETER
# =====================
file_excel = file_path
target_kdkab = "14"                                                                # code of the regency/city to be calculated

# =====================
# READ DATA
# =====================
df = pd.read_excel(
    file_excel,
    sheet_name="logikadasar",
    dtype=str
)

df_pml = pd.read_excel(
    file_excel,
    sheet_name="PML",
    dtype=str
)

# =====================
# DATA CLEANING
# =====================
df["Petugas revisi"] = pd.to_numeric(df["Petugas revisi"], errors="coerce").fillna(0)

# if there is a kdkab column in logikadasar
if "kdkab" in df.columns:
    df_kab = df[df["kdkab"] == target_kdkab].copy()

else:
    # if there is no kdkab, the regency code is usually found before kdkec
    df["kdkec"] = df["kdkec"].str.zfill(3)
    df_kab = df[df["kdkec"].str.startswith(target_kdkab[-1])].copy()

# =====================
# GET REGENCY DATA
# =====================
row_kab = df_pml[df_pml["kdkab"] == target_kdkab].iloc[0]

nama_kab = row_kab["Kab/Kota"]
jumlah_pml = int(row_kab["Perkiraan Jumlah PML"])

# =====================
# CALCULATE TOTAL STAFF
# =====================
total_petugas = int(df_kab["Petugas revisi"].sum())

print("================================")
print("Kab/Kota :", nama_kab)
print("Total petugas :", total_petugas)
print("Total PML :", jumlah_pml)
print("================================")

print("\nJumlah desa :", len(df_kab))

# =====================
# AVERAGE STAFF PER PML
# =====================
target = total_petugas / jumlah_pml
print("\nRata-rata petugas per PML :", round(target,2))

# =====================
# PREVIEW DATA
# =====================
print("\nPreview data:")
print(df_kab[["kdkec","kddesa","Petugas revisi"]].head())

In [ ]:
print(jumlah_pml)
print(total_petugas)

In [ ]:
# =====================
# TARGET NUMBER OF STAFF PER PML
# =====================
base = total_petugas // jumlah_pml
sisa = total_petugas % jumlah_pml

target_global = [base+1]*sisa + [base]*(jumlah_pml-sisa)

# =====================
# CALCULATE STAFF PER SUBDISTRICT
# =====================
petugas_kec = (
    df_kab.groupby("kdkec")["Petugas revisi"]
    .sum()
    .reset_index()
)

petugas_kec["pml_kec"] = (
    petugas_kec["Petugas revisi"] / total_petugas * jumlah_pml
)

petugas_kec["pml_kec"] = petugas_kec["pml_kec"].round().astype(int)

petugas_kec.loc[petugas_kec["pml_kec"] < 1, "pml_kec"] = 1                        # Ensure at least 1 PML per subdistrict

selisih = jumlah_pml - petugas_kec["pml_kec"].sum()                               # Adjust the number of PMLs so they match

if selisih != 0:
    idx = petugas_kec["Petugas revisi"].idxmax()
    petugas_kec.loc[idx, "pml_kec"] += selisih


# =====================
# STAFF ALLOCATION
# =====================
hasil = []
pml_counter = 1

for _, kec_row in petugas_kec.iterrows():

    kec = kec_row["kdkec"]
    pml_kec = int(kec_row["pml_kec"])

    df_kec = df_kab[df_kab["kdkec"] == kec].copy()

    total_kec = int(df_kec["Petugas revisi"].sum())

    base_kec = total_kec // pml_kec
    sisa_kec = total_kec % pml_kec

    kapasitas = [base_kec+1]*int(sisa_kec) + [base_kec]*(pml_kec-int(sisa_kec))

    pml_list = []                                                                  # create a list of PMLs
    for i in range(pml_kec):
        pml_list.append({
            "pml": pml_counter+i,
            "sisa": kapasitas[i]
        })

    df_kec = df_kec.sort_values("Petugas revisi", ascending=False)                # sort by the largest number of villages

    for _, desa_row in df_kec.iterrows():

        desa = desa_row["kddesa"]
        petugas = int(desa_row["Petugas revisi"])

        while petugas > 0:

            pml_list = sorted(pml_list, key=lambda x: x["sisa"], reverse=True)    # sort PMLs by the largest capacity

            for pml in pml_list:

                if pml["sisa"] == 0:
                    continue

                ambil = min(petugas, pml["sisa"])

                hasil.append({
                    "PML": pml["pml"],
                    "kdkec": kec,
                    "kddesa": desa,
                    "petugas": ambil
                })

                pml["sisa"] -= ambil
                petugas -= ambil

                if petugas == 0:
                    break

    pml_counter += pml_kec

In [ ]:
# =====================
# DATAFRAME RESULTS
# =====================
df_hasil = pd.DataFrame(hasil)

# =====================
# SUMMARY
# =====================
rekap_pml = df_hasil.groupby("PML")["petugas"].sum().reset_index()

rekap_kec = df_hasil.groupby("kdkec")["petugas"].sum().reset_index()

In [ ]:
# =====================
# PREVIEW OUTPUT
# =====================
print("\n================ PREVIEW HASIL ================\n")

print("Detail pembagian (10 baris pertama):")
print(df_hasil.head(10))

print("\nRekap PML:")
print(rekap_pml)

print("\nRekap Kecamatan:")
print(rekap_kec)

print("\nTotal petugas hasil pembagian:", df_hasil["petugas"].sum())
print("Total petugas awal:", total_petugas)

print("\nJumlah PML terbentuk:", df_hasil["PML"].nunique())
print("Target jumlah PML:", jumlah_pml)

In [ ]:
print("Detail pembagian (10 baris pertama):")
print(df_hasil.tail(10))

In [ ]:
# =====================
# SAVE OUTPUT
# =====================
output_file = os.path.join(
    os.path.dirname(file_excel),
    f"{target_kdkab} pembagian_PML_{nama_kab}_optimal.xlsx"
)

with pd.ExcelWriter(output_file) as writer:

    df_hasil.to_excel(
        writer,
        sheet_name="detail_pembagian",
        index=False
    )

    rekap_pml.to_excel(
        writer,
        sheet_name="rekap_PML",
        index=False
    )

    rekap_kec.to_excel(
        writer,
        sheet_name="rekap_kecamatan",
        index=False
    )

print("Output berhasil dibuat:", output_file)